In [1]:
!rm -rf llm_pii_anonymizer_fr
!git clone https://github.com/ellakonan41/llm_pii_anonymizer_fr.git
import sys
sys.path.append("llm_pii_anonymizer_fr")

from src.evaluation import evaluer, evaluer_exemple
from src.prompt import construire_messages

Cloning into 'llm_pii_anonymizer_fr'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 40 (delta 16), reused 34 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 36.10 KiB | 6.02 MiB/s, done.
Resolving deltas: 100% (16/16), done.


In [2]:
import glob, json

chemin = glob.glob("/kaggle/input/**/test.jsonl", recursive=True)[0]
with open(chemin, encoding="utf-8") as f:
    test = [json.loads(ligne) for ligne in f]
print(chemin, len(test))

/kaggle/input/notebooks/konanella/02-preparation-donnees/test.jsonl 500


In [3]:
controle = evaluer(test, [ex["masked_text"] for ex in test])
for cle, valeur in controle.items():
    if cle != "protection_par_type":
        print(f"{cle:<25} {valeur:.1%}")

protection                100.0%
textes_sans_fuite         100.0%
preservation              100.0%
correspondance_exacte     100.0%


## 4)charger le modèle

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

NOM_MODELE = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(NOM_MODELE, torch_dtype=torch.float16, device_map="cuda")
model.eval()
print(f"{model.num_parameters() / 1e6:.0f} M paramètres")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

494 M paramètres


## 5) voir ce que le modèle reçoit vraiment

In [5]:
exemple = test[0]["source_text"]
prompt = tokenizer.apply_chat_template(construire_messages(exemple), tokenize=False, add_generation_prompt=True)
print(prompt)

<|im_start|>system
Tu es un outil d'anonymisation de textes. Réécris le texte fourni en remplaçant chaque donnée personnelle par son type entre crochets, choisi parmi : [GIVENNAME], [SURNAME], [TELEPHONENUM], [CITY], [TIME], [EMAIL], [DATE], [STREET], [BUILDINGNUM], [IDCARDNUM], [TITLE], [AGE], [ZIPCODE], [PASSPORTNUM], [SEX], [TAXNUM], [SOCIALNUM], [CREDITCARDNUMBER], [DRIVERLICENSENUM], [GENDER]. Ne modifie rien d'autre dans le texte. Réponds uniquement avec le texte anonymisé.<|im_end|>
<|im_start|>user
18:26:13 Ilya Selhida: 'Bien sûr, Metihe Rebeca ! Je suis Tshibola Rolider et je suis impatient de voir le résultat final de notre projet de carte.'<|im_end|>
<|im_start|>assistant



## 6)la fonction de génération

In [6]:
from tqdm.auto import tqdm

@torch.no_grad()
def anonymiser(textes, taille_lot=16, max_new_tokens=512):
    sorties = []
    for i in tqdm(range(0, len(textes), taille_lot)):
        lot = textes[i : i + taille_lot]
        prompts = [
            tokenizer.apply_chat_template(construire_messages(t), tokenize=False, add_generation_prompt=True)
            for t in lot
        ]
        entrees = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        generes = model.generate(**entrees, max_new_tokens=max_new_tokens, do_sample=False)
        nouveaux = generes[:, entrees["input_ids"].shape[1]:]
        sorties += [s.strip() for s in tokenizer.batch_decode(nouveaux, skip_special_tokens=True)]
    return sorties

## 7)essai sur 3 textes

In [7]:
essai = anonymiser([ex["source_text"] for ex in test[:3]])
for ex, sortie in zip(test[:3], essai):
    print("ORIGINAL :", ex["source_text"])
    print("ATTENDU  :", ex["masked_text"])
    print("MODÈLE   :", sortie)
    print("-" * 80)

  0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ORIGINAL : 18:26:13 Ilya Selhida: 'Bien sûr, Metihe Rebeca ! Je suis Tshibola Rolider et je suis impatient de voir le résultat final de notre projet de carte.'
ATTENDU  : [TIME] [GIVENNAME]: 'Bien sûr, [GIVENNAME] ! Je suis [GIVENNAME] [SURNAME] et je suis impatient de voir le résultat final de notre projet de carte.'
MODÈLE   : Ilya Selhida: 'Bien sûr, Mirehe Rebeca ! Je suis Tshibola Rolider et je suis impatient de voir le résultat final de notre projet de carte.'
--------------------------------------------------------------------------------
ORIGINAL : Bonjour, je suis Sherap Iblikci et j'aimerais commander un puzzle qui correspond à mes goûts personnels.
ATTENDU  : Bonjour, je suis [GIVENNAME] [SURNAME] et j'aimerais commander un puzzle qui correspond à mes goûts personnels.
MODÈLE   : **Sherap Iblikci**  
**Commande**  
**Puzzle**  
**Goût**  
**Personnel**
--------------------------------------------------------------------------------
ORIGINAL : Disposez-vous d'une assurance re

## 8) Evaluation complète

In [8]:
sorties_v0 = anonymiser([ex["source_text"] for ex in test])

  0%|          | 0/32 [00:00<?, ?it/s]

In [9]:
resultats_v0 = evaluer(test, sorties_v0)

In [10]:
for cle, valeur in resultats_v0.items():
    if cle != "protection_par_type":
        print(f"{cle:<25} {valeur:.1%}")
print("\nProtection par type :")
for label, taux in resultats_v0["protection_par_type"].items():
    print(f"  {label:<20} {taux:.1%}")

protection                42.9%
textes_sans_fuite         39.8%
preservation              27.2%
correspondance_exacte     0.0%

Protection par type :
  GIVENNAME            42.1%
  SURNAME              34.1%
  TELEPHONENUM         56.0%
  CITY                 31.3%
  TIME                 50.7%
  EMAIL                52.4%
  STREET               24.1%
  DATE                 52.8%
  BUILDINGNUM          38.3%
  TITLE                50.0%
  IDCARDNUM            57.7%
  AGE                  48.0%
  ZIPCODE              29.4%
  CREDITCARDNUMBER     41.7%
  PASSPORTNUM          75.0%
  TAXNUM               30.0%
  DRIVERLICENSENUM     70.0%
  SEX                  37.5%
  SOCIALNUM            42.9%
  GENDER               0.0%


## 9)sauvegarder

In [11]:
with open("/kaggle/working/sorties_v0.jsonl", "w", encoding="utf-8") as f:
    for ex, sortie in zip(test, sorties_v0):
        f.write(json.dumps({"uid": ex["uid"], "sortie": sortie}, ensure_ascii=False) + "\n")

In [12]:
with open("/kaggle/working/resultats_v0.json", "w", encoding="utf-8") as f:
    json.dump(resultats_v0, f, ensure_ascii=False, indent=2)

In [13]:
import re

avec_etiquette = sum(bool(re.search(r"\[[A-Z]+\]", s)) for s in sorties_v0)
tres_courts = sum(len(s) < 0.5 * len(ex["source_text"]) for ex, s in zip(test, sorties_v0))

print(f"Sorties avec au moins une étiquette [..] : {avec_etiquette / len(test):.1%}")
print(f"Sorties 2 fois plus courtes que l'original : {tres_courts / len(test):.1%}")

Sorties avec au moins une étiquette [..] : 14.8%
Sorties 2 fois plus courtes que l'original : 19.6%
